In [1]:
import pathlib
from typing import Sequence, Callable, List

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import equinox as eqx
from jaxtyping import Array, Float, Int, PyTree, PRNGKeyArray

import torch
from torch.utils.data import DataLoader, Subset

import altair as alt

import trapallada.data
import trapallada.model.signal

# import ipdb

# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# input_directory = pathlib.Path.cwd().parent / "light curves" / "Simulations"
# input_directory = pathlib.Path.cwd().parent / "light curves"
input_directory = pathlib.Path.cwd()
assert input_directory.exists()

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [5]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

12983

In [6]:
rng_key = jr.PRNGKey(42)

parameters = {
    'training': {
        'splitting': {
            'test_size':  0.2,
            'random_state': 42
        },
        'batch_size': 32,
        'learning_rate': 1e-3,
    },
}

In [7]:
train_idx, validation_idx = train_test_split(
    np.arange(len(ds)), 
    **parameters['training']['splitting'], 
    shuffle=True,
    stratify=ds.labels
)
train_ds, validation_ds = Subset(ds, train_idx), Subset(ds, validation_idx)

# a random key...
key, subkey = jr.split(rng_key)

# ...to compute a seed for a PyTorch `Generator`...
torch_seed = jr.bits(subkey, (1,), dtype=jnp.uint32).item()
torch_rng = torch.Generator().manual_seed(torch_seed)

# ...meant for "batching" in the `DataLoader`'s
train_dl = DataLoader(train_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)
validation_dl = DataLoader(validation_ds, batch_size=parameters['training']['batch_size'], shuffle=False, generator=torch_rng)

In [8]:
batch = next(iter(train_dl))

In [9]:
signals_len = ds[0][1].shape[-1]
signals_len

10000

In [10]:
tuple(ds[0][1].shape)

(1, 10000)

In [11]:
parameters['model'] = {
    'input_size': ds[0][1].shape,
    'dim_z': 4,
    'kernel_sizes': [128, 64, 32],
    'channels': [12, 12, 12],
    'stride': 2,
}

In [35]:
class Encoder(eqx.Module):

    layers: list
    # linear_in_features: int
    layers_spatial_sizes: List[int]

    def __init__(self, input_size: Sequence[int], dim_z: int, kernel_sizes: List[int], channels: List[int], stride: int, key: PRNGKeyArray):

        padding = 0

        # for the sake of readability
        n_input_maps, input_maps_len = input_size
        print(f'{n_input_maps=}, {input_maps_len=}')

        # the number of input maps is the number of input channels
        channels = [n_input_maps] + channels

        self.layers = []

        conv_output_len = input_maps_len

        self.layers_spatial_sizes = [input_maps_len]

        for in_channels, out_channels, key, kernel_size in zip(
            channels[:-1], channels[1:],
            jr.split(key, len(channels) - 1),
            kernel_sizes
        ):
            
            # the length of the feature map for the next layer
            conv_output_len = trapallada.util.conv_output_size(conv_output_len, (kernel_size,), (stride,), (padding,))
            print(f'{conv_output_len=}')

            key, subkey = jr.split(key)
            
            self.layers.extend([
                eqx.nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, key=subkey),
                jax.nn.relu,
                eqx.nn.LayerNorm((out_channels, conv_output_len[0]))
            ])

            self.layers_spatial_sizes.append(conv_output_len[0])

        
        # self.linear_in_features = out_channels * conv_output_len[0]

        self.layers_spatial_sizes.append(out_channels * conv_output_len[0])

        self.layers.extend([
            jnp.ravel,
            # eqx.nn.Linear(in_features=self.linear_in_features, out_features=2*dim_z, key=key),
            eqx.nn.Linear(in_features=self.layers_spatial_sizes[-1], out_features=2*dim_z, key=key),
            # lambda x: x.at[dim_z:].set(jax.nn.softplus(x[dim_z:]) + 1e-3)
            lambda x: x.at[dim_z:].set(jax.nn.softplus(x[dim_z:]))
        ])
        
    def __call__(self, x: Float[Array, '1 length']) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)

        return x

    def debug_call(self, x: Float[Array, '1 length']) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)

            print(f'after {layer.__module__}: {x.shape}')

In [36]:
x = batch[1][0].numpy()
model = Encoder(**parameters['model'], key=jr.PRNGKey(42))
model.debug_call(x)
print(model)

n_input_maps=1, input_maps_len=10000
conv_output_len=(4937,)
conv_output_len=(2437,)
conv_output_len=(1203,)
after equinox.nn._conv: (12, 4937)
after jax._src.nn.functions: (12, 4937)
after equinox.nn._normalisation: (12, 4937)
after equinox.nn._conv: (12, 2437)
after jax._src.nn.functions: (12, 2437)
after equinox.nn._normalisation: (12, 2437)
after equinox.nn._conv: (12, 1203)
after jax._src.nn.functions: (12, 1203)
after equinox.nn._normalisation: (12, 1203)
after jax._src.numpy.lax_numpy: (14436,)
after equinox.nn._linear: (8,)
after __main__: (8,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,1,128],
      bias=f32[12,1],
      in_channels=1,
      out_channels=12,
      kernel_size=(128,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 4937),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,4937],

In [37]:
model.layers_spatial_sizes

[10000, 4937, 2437, 1203, 14436]

In [34]:
model(x)

Array([-0.1810234 ,  0.02323518, -0.00103099,  0.33093098,  0.49362022,
        0.81390536,  0.6616099 ,  0.72323906], dtype=float32)

In [38]:
            # eqx.nn.Linear(dim_z, 64, key=key1),
            # jax.nn.relu,
            # eqx.nn.Linear(64, 512, key=key2),
            # jax.nn.sigmoid,
            # eqx.nn.Linear(512, conv_input_size, key=key3),
            # jax.nn.relu,
            # lambda x: jnp.reshape(x, (40, 2, 14)), # from a 1D vector to a 3D array
            # eqx.nn.ConvTranspose2d(40, 20, kernel_size=4, stride=1, padding=1, key=key4),
            # jax.nn.relu,
            # eqx.nn.ConvTranspose2d(20, 20, kernel_size=4, stride=(2, 2), padding=1, output_padding=(1, 1), key=key5),
            # jax.nn.relu,
            # eqx.nn.ConvTranspose2d(20, 10, kernel_size=4, stride=(2, 2), padding=1, key=key6),
            # jax.nn.relu,
            # eqx.nn.ConvTranspose2d(10, 10, kernel_size=4, stride=(2, 2), padding=1, output_padding=(1, 1), key=key7),
            # jax.nn.relu,
            # eqx.nn.ConvTranspose2d(10, 1, kernel_size=4, stride=(1, 2), padding=1, key=key8),
            # # jax.nn.sigmoid
            # jnp.tanh

In [39]:
# parameters['model'] = {
#     'input_size': ds[0][1].shape,
#     'dim_z': 4,
#     'kernel_sizes': [128, 64, 32],
#     'channels': [12, 12, 12],
#     'stride': 2,
# }

In [40]:
del parameters['model']['input_size']
parameters['model']['layers_spatial_sizes'] = model.layers_spatial_sizes
parameters['model']

{'dim_z': 4,
 'kernel_sizes': [128, 64, 32],
 'channels': [12, 12, 12],
 'stride': 2,
 'layers_spatial_sizes': [10000, 4937, 2437, 1203, 14436]}

In [58]:
class Decoder(eqx.Module):

    layers: list

    def __init__(self, dim_z: int, layers_spatial_sizes: List[int], kernel_sizes: List[int], channels: List[int], stride: int, key: PRNGKeyArray):

        padding = 0

        channels = list(reversed(channels)) + [1]
        kernel_sizes = list(reversed(kernel_sizes))
        layers_spatial_sizes = list(reversed(layers_spatial_sizes))

        self.layers = []

        # conv_output_len = linear_out_features // channels[0]
        # print(conv_output_len)

        self.layers.extend([
            eqx.nn.Linear(in_features=dim_z, out_features=layers_spatial_sizes[0], key=key),
            jax.nn.relu,
            # CAVEAT: `-1` should be equivalent to `conv_output_len` but (because of lazy evaluation?) it's not because it's actually evaluated for the final value of `conv_output_len`
            lambda x: jnp.reshape(x, (channels[0], -1)), # from a 1D vector to a 2D array
            # lambda x: jnp.reshape(x, (channels[0], conv_output_len)), # from a 1D vector to a 2D array
        ])
        # print((channels[0], conv_output_len))

        # after the `reshape`ing
        conv_output_len = layers_spatial_sizes[1]

        for in_channels, out_channels, key, kernel_size, expected_len in zip(
            channels[:-1], channels[1:],
            jr.split(key, len(channels) - 1),
            kernel_sizes,
            layers_spatial_sizes[2:],
        ):
            

            conv_output_len = trapallada.util.transp_conv_output_size(conv_output_len, (kernel_size,), (stride,), (padding,))
            print(f'{conv_output_len=}, {expected_len=}')

            output_padding = expected_len - conv_output_len[0]
            
            key, subkey = jr.split(key)
            
            self.layers.extend([
                eqx.nn.ConvTranspose1d(
                    in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride,
                    # output_padding=output_padding, key=subkey),
                    output_padding=output_padding, key=subkey),
                jax.nn.relu,
                # eqx.nn.LayerNorm((out_channels, -1))
            ])

            conv_output_len += output_padding
        
    def __call__(self, x: Float[Array, '1 length']) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)

        return x

    def debug_call(self, x: Float[Array, '1 length']) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)

            print(f'after {layer.__module__}: {x.shape}')

In [59]:
x = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
model = Decoder(**parameters['model'], key=jr.PRNGKey(42))
model.debug_call(x)

conv_output_len=(2436,), expected_len=2437
conv_output_len=(4936,), expected_len=4937
conv_output_len=(10000,), expected_len=10000
after equinox.nn._linear: (14436,)
after jax._src.nn.functions: (14436,)
after __main__: (12, 1203)
after equinox.nn._conv: (12, 2437)
after jax._src.nn.functions: (12, 2437)
after equinox.nn._conv: (12, 4937)
after jax._src.nn.functions: (12, 4937)
after equinox.nn._conv: (1, 10000)
after jax._src.nn.functions: (1, 10000)
